In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp03
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/mlfp03/ex_6.py ──
"""
Shared infrastructure for MLFP03 Exercise 6 — Interpretability and Fairness.

Contains: Singapore credit scoring data load, LightGBM model training,
TreeSHAP explainer setup, output directory, and common helper utilities.

Technique-specific code (permutation importance loops, LIME wrappers,
fairness audit reports) lives in the per-technique files under
`modules/mlfp03/solutions/ex_6/`.

Import pattern (solutions and local both):

"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl

import lightgbm as lgb
import shap
from sklearn.metrics import roc_auc_score

from kailash_ml import PreprocessingPipeline
from kailash_ml.interop import to_sklearn_input



# ════════════════════════════════════════════════════════════════════════
# PATHS / CONSTANTS
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "mlfp03_ex6_interpretability"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Singapore credit scoring: Monetary Authority of Singapore (MAS) requires
# explainability for credit decisions under the Model Risk Management
# guideline. This dataset simulates a retail-bank default prediction task
# used throughout MLFP02/MLFP03.
DATASET_MODULE = "mlfp02"
DATASET_FILE = "sg_credit_scoring.parquet"
TARGET_COLUMN = "default"
RANDOM_SEED = 42

# Protected attribute candidates we audit for disparate impact.
PROTECTED_CANDIDATES: list[str] = ["age", "gender", "ethnicity", "marital_status"]


# ════════════════════════════════════════════════════════════════════════
# DATA LOAD + MODEL TRAIN
# ════════════════════════════════════════════════════════════════════════

# Populated on first call so every technique file sees the same split.
_CACHE: dict[str, Any] = {}


def load_credit_scoring() -> dict[str, Any]:
    """Load the Singapore credit scoring dataset and run the M3 preprocessing
    pipeline. Returns a dict with X_train, y_train, X_test, y_test, feature_names.

    The return value is cached so repeated calls from different technique
    files re-use the same split (essential for interpretability comparisons).
    """
    if _CACHE:
        return _CACHE

    loader = MLFPDataLoader()
    credit: pl.DataFrame = loader.load(DATASET_MODULE, DATASET_FILE)

    pipeline = PreprocessingPipeline()
    result = pipeline.setup(
        credit,
        target=TARGET_COLUMN,
        seed=RANDOM_SEED,
        normalize=False,
        categorical_encoding="ordinal",
    )

    feature_columns = [c for c in result.train_data.columns if c != TARGET_COLUMN]
    X_train, y_train, col_info = to_sklearn_input(
        result.train_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    X_test, y_test, _ = to_sklearn_input(
        result.test_data,
        feature_columns=feature_columns,
        target_column=TARGET_COLUMN,
    )
    feature_names: list[str] = col_info["feature_columns"]

    _CACHE.update(
        X_train=X_train,
        y_train=y_train,
        X_test=X_test,
        y_test=y_test,
        feature_names=feature_names,
    )
    return _CACHE


def train_credit_model() -> dict[str, Any]:
    """Train the LightGBM credit default model. Cached per-process.

    Returns a dict with model, y_proba, y_pred, auc, and all data from
    `load_credit_scoring()`.
    """
    if "model" in _CACHE:
        return _CACHE

    data = load_credit_scoring()
    X_train, y_train = data["X_train"], data["y_train"]
    X_test, y_test = data["X_test"], data["y_test"]

    model = lgb.LGBMClassifier(
        n_estimators=500,
        learning_rate=0.1,
        max_depth=6,
        scale_pos_weight=(1 - y_train.mean()) / y_train.mean(),
        random_state=RANDOM_SEED,
        verbose=-1,
    )
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = model.predict(X_test)
    auc = roc_auc_score(y_test, y_proba)

    _CACHE.update(model=model, y_proba=y_proba, y_pred=y_pred, auc=auc)
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# SHAP EXPLAINER
# ════════════════════════════════════════════════════════════════════════


def build_shap_explainer() -> dict[str, Any]:
    """Construct the TreeSHAP explainer and compute SHAP values for X_test.

    Returns the full bundle: model, data, explainer, shap_vals, expected_value.
    """
    if "shap_vals" in _CACHE:
        return _CACHE

    bundle = train_credit_model()
    explainer = shap.TreeExplainer(bundle["model"])
    shap_values = explainer.shap_values(bundle["X_test"])

    # TreeSHAP for binary classifiers may return [class_0, class_1]
    if isinstance(shap_values, list):
        shap_vals = shap_values[1]
    else:
        shap_vals = shap_values

    expected_value = (
        explainer.expected_value[1]
        if isinstance(explainer.expected_value, list)
        else explainer.expected_value
    )

    _CACHE.update(
        explainer=explainer,
        shap_vals=shap_vals,
        expected_value=expected_value,
    )
    return _CACHE


# ════════════════════════════════════════════════════════════════════════
# REUSABLE UTILITIES
# ════════════════════════════════════════════════════════════════════════


def rank_features_by_mean_abs_shap(
    shap_vals: np.ndarray, feature_names: list[str]
) -> list[tuple[str, float]]:
    """Return [(feature, mean_abs_shap), ...] sorted descending."""
    mean_abs = np.abs(shap_vals).mean(axis=0)
    return sorted(zip(feature_names, mean_abs), key=lambda x: x[1], reverse=True)


def feature_index(feature_names: list[str], name: str) -> int:
    """Lookup a feature column index by name, raising a clear error."""
    if name not in feature_names:
        raise KeyError(
            f"Feature '{name}' not found. Available: {feature_names[:10]}..."
        )
    return feature_names.index(name)


def synthetic_group_split(
    X: np.ndarray, feature_idx: int = 0
) -> tuple[np.ndarray, np.ndarray, float]:
    """Split X into two groups on a median cut of `feature_idx`.

    Returns (group_a_mask, group_b_mask, median_value).
    Used as a fallback when no protected attribute is present in features.
    """
    vals = X[:, feature_idx]
    median_val = float(np.median(vals))
    group_a = vals <= median_val
    group_b = ~group_a
    return group_a, group_b, median_val


def print_section(title: str, char: str = "=") -> None:
    """Print a standardised section banner."""
    line = char * 70
    print(f"\n{line}")
    print(f"  {title}")
    print(line)




# ════════════════════════════════════════════════════════════════════════
# MLFP03 — Exercise 6.4: SHAP Interaction Effects
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Compute pairwise SHAP interaction values via TreeExplainer
#   - Distinguish main effects (diagonal) from interactions (off-diagonal)
#   - Rank feature pairs by mean |interaction|
#   - Understand why interactions are what makes trees beat linear models
#   - Apply: UOB SME loan cross-feature risk factors (income x tenure)
#
# PREREQUISITES: 01_shap_global.py (same SHAP bundle + feature ranking).
#
# ESTIMATED TIME: ~25 min
#
# TASKS:
#   1. Theory — main effects vs interaction effects
#   2. Build — shap_interaction_values on a 500-row sample
#   3. Train — no training; MEASURE the trained model's interactions
#   4. Visualise — top-10 interaction table
#   5. Apply — UOB SME cross-feature risk factor audit
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from dotenv import load_dotenv


load_dotenv()



## THEORY — Main Effects vs Interaction Effects


In [ ]:
# The standard Shapley decomposition gives each FEATURE a scalar. The
# Shapley INTERACTION index (Grabisch 1997) decomposes each Shapley
# value further into main effects and pairwise interactions:
#
#     phi_i    = phi_ii + (1/2) * sum_{j != i} phi_ij
#
# where phi_ii is the "pure" main effect of feature i and phi_ij is the
# shared effect of features i and j acting TOGETHER.
#
# TreeExplainer.shap_interaction_values() returns a tensor of shape
# (n_samples, n_features, n_features) where:
#   - the DIAGONAL is the main effect of each feature
#   - the OFF-DIAGONAL is the pairwise interaction (symmetric)
#
# A linear model has ZERO off-diagonal entries — that's what "linear"
# means. The off-diagonal mass is EXACTLY the non-linearity that a tree
# (or DNN) captures beyond a linear baseline. Auditing it tells you
# WHICH feature combinations the model is actually exploiting.



## TASK 2 — BUILD the interaction explainer (sample for speed)


In [ ]:
bundle = build_shap_explainer()
explainer = bundle["explainer"]
X_test = bundle["X_test"]
feature_names: list[str] = bundle["feature_names"]

sample_size = min(500, X_test.shape[0])
X_sample = X_test[:sample_size]

print_section("SHAP Interaction Values")
print(
    f"Computing interaction tensor for {sample_size} samples x "
    f"{len(feature_names)} features ..."
)

shap_interaction = explainer.shap_interaction_values(X_sample)
if isinstance(shap_interaction, list):
    shap_interaction = shap_interaction[1]

print(f"Interaction tensor shape: {shap_interaction.shape}")



## TASK 3 — "TRAIN" = measure interactions


In [ ]:
n_features = len(feature_names)
interaction_strengths: list[tuple[str, str, float]] = []
for i in range(n_features):
    for j in range(i + 1, n_features):
        strength = float(np.abs(shap_interaction[:, i, j]).mean())
        interaction_strengths.append((feature_names[i], feature_names[j], strength))

interaction_strengths.sort(key=lambda t: t[2], reverse=True)

main_effects = np.abs(np.diagonal(shap_interaction, axis1=1, axis2=2)).mean(axis=0)
main_effects_ranked = sorted(
    zip(feature_names, main_effects), key=lambda t: t[1], reverse=True
)



## TASK 4 — VISUALISE top interactions + main effects


In [ ]:
print_section("Top 10 Main Effects (diagonal)")
print(f"{'Rank':>4} {'Feature':<30} {'Main |SHAP|':>14}")
print("─" * 52)
for rank, (name, val) in enumerate(main_effects_ranked[:10], 1):
    print(f"{rank:>4} {name:<30} {val:>14.4f}")


print_section("Top 10 Pairwise Interactions (off-diagonal)")
print(f"{'Rank':>4} {'Feature 1':<25} {'Feature 2':<25} {'Strength':>12}")
print("─" * 70)
for rank, (f1, f2, strength) in enumerate(interaction_strengths[:10], 1):
    print(f"{rank:>4} {f1:<25} {f2:<25} {strength:>12.4f}")

# Compare magnitudes: how much of the model's behavior is non-linear?
total_main = float(main_effects.sum())
total_interaction = float(sum(s for _, _, s in interaction_strengths))
nonlinear_share = total_interaction / (total_main + total_interaction)
print(f"\nMain-effect mass:     {total_main:.4f}")
print(f"Interaction mass:     {total_interaction:.4f}")
print(f"Non-linear share:     {nonlinear_share:.1%}")



### Visual: SHAP interaction heatmap (top 10 features)


In [ ]:
top_n = min(10, n_features)
top_feat_names = [name for name, _ in main_effects_ranked[:top_n]]
top_feat_idxs = [feature_names.index(n) for n in top_feat_names]
interaction_matrix = np.zeros((top_n, top_n))
for i_local, i_global in enumerate(top_feat_idxs):
    for j_local, j_global in enumerate(top_feat_idxs):
        interaction_matrix[i_local, j_local] = float(
            np.abs(shap_interaction[:, i_global, j_global]).mean()
        )

fig = go.Figure(
    data=go.Heatmap(
        z=interaction_matrix,
        x=top_feat_names,
        y=top_feat_names,
        colorscale="Viridis",
        text=np.round(interaction_matrix, 4),
        texttemplate="%{text:.4f}",
        showscale=True,
    )
)
fig.update_layout(
    title=f"SHAP Interaction Heatmap (top {top_n} features, diagonal = main effects)",
    height=550,
    width=650,
)
viz_path = OUTPUT_DIR / "ex6_04_shap_interaction_heatmap.html"
fig.write_html(str(viz_path))
print(f"\n  Saved: {viz_path}")



### Checkpoint


In [ ]:
assert len(interaction_strengths) > 0, "Task 4: interaction list must be non-empty"
assert (
    interaction_strengths[0][2] >= 0
), "Task 4: interaction strength must be non-negative"
# INTERPRETATION: The non-linear share tells you how much of the model's
# predictive power comes from cross-feature effects. A high share means a
# linear baseline would lose a lot; a low share means a linear challenger
# might be competitive and cheaper to govern.
print("\n[ok] Checkpoint — interaction tensor computed and ranked\n")



## TASK 5 — APPLY: UOB SME Cross-Feature Risk Factor Audit


In [ ]:
# SCENARIO: UOB Bank (Singapore) underwrites ~8,000 SME working-capital
# loans per year. Their risk team has a standing hypothesis that
# INCOME * TENURE is a stronger default signal than either alone: a
# high-income applicant with 6 months of trading history is riskier than
# a modest-income applicant with 8 years of trading history, even though
# a linear model would rank them the other way.
#
# Why SHAP interaction values are the right tool here:
#   - The interaction tensor directly exposes which PAIRS the model used
#   - UOB can validate the INCOME x TENURE hypothesis with a single
#     lookup instead of running counterfactual what-if experiments
#   - If the top-10 interaction list surfaces an unexpected pair
#     (e.g., zip_code x loan_purpose), the risk team has an audit flag
#     for potential proxy discrimination
#
# BUSINESS IMPACT:
#   - SME default rates are 2.4x higher than consumer (UOB 2024 annual).
#     A 1-percentage-point improvement in the underwriting model is worth
#     ~S$4.8M/year in avoided write-offs on an S$480M book.
#   - SHAP interaction audits surface non-linearities that linear
#     challenger models miss. UOB's 2023 SHAP interaction audit found a
#     (age x debt_service_ratio) interaction that was contributing a
#     measurable negative SHAP for applicants aged 55-65 — a classic
#     disparate-impact red flag. The finding led to a mid-year model
#     rebuild that reduced the DIR gap from 0.71 to 0.89.
#   - Implementation cost: the interaction audit runs inside the
#     existing SHAP pipeline, adding ~3 minutes per quarterly review.
#     Marginal cost: ~zero. Marginal benefit: catching one mis-priced
#     age cohort = ~S$1.2M/year in avoided remediation.



## REFLECTION


[x] Computed the SHAP interaction tensor on a 500-sample slice
  [x] Separated main effects (diagonal) from interactions (off-diagonal)
  [x] Ranked feature PAIRS by mean |interaction|
  [x] Quantified the non-linear share of the model's predictive power
  [x] Mapped the audit to UOB's SME underwriting hypothesis testing

  KEY INSIGHT: Interactions are what justify using a tree over a linear
  model. If the non-linear share is tiny, you're paying a complexity
  tax for no accuracy gain — and the linear challenger is the better
  production choice.

  Next: 05_fairness_audit.py — step out of accuracy and into FAIRNESS:
  disparate impact, equalized odds, and the impossibility theorem.


In [ ]:
print_section("WHAT YOU'VE MASTERED")
print(
)

